# Handle communities

In [1]:
import pandas as pd
import numpy as np
import torch

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
# column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv("archive/ifashion_data.csv")
df.columns = ['userId','movieId','rating']
len(df)

In [3]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)
df = train_data

In [ ]:
communities = []
user = set()
with open('archive/ifashion.txt') as f:
    lines = f.readlines()
    for c in lines:
        flag = 0
        c.strip()
        c = c.split()
        c = c[1:]
        c = [int(x) for x in c]
        for u in c: 
            if(u not in user):
                user.add(u)
                flag = 1
        if(flag): communities.append(c)
com_num = len(communities)
com_num

In [ ]:
# find the community of user:
def find_community(user):
    com = []
    for i in range(com_num):
        if user in communities[i]:
            return i
#     return com
find_community(89)

In [ ]:
for idx, i in train_data.iterrows():
    if i['userId'] in communities[0]:
        if i['movieId'] == 14668:
            print(i['userId'])

In [7]:
def create_matrix(train_data):
#     user_rating_avg = train_data.groupby('user_id')['rating'].mean()
#     user_rating_avg
#     # Create the pivot table
#     train_data_matrix = train_data.pivot_table(index='user_id', columns='item_id', values='rating')
#     # Fill missing values with user averages
#     train_data_matrix_avg_fill = train_data_matrix.apply(lambda x: x.fillna(user_rating_avg[x.name]), axis=1)
#     return train_data_matrix_avg_fill

    train_data_matrix_0_fill = train_data.pivot_table(index='userId', columns='movieId', values='rating', fill_value=0)
    
    return train_data_matrix_0_fill

In [8]:
import numpy as np

def calculate_similarity_between_users(user1, user2, similarity_type='euclidean'):
    if similarity_type == 'euclidean':
        # Calculate the Euclidean distance
        distance = np.sqrt(np.nansum((user1 - user2) ** 2))
        # Convert distance to similarity
        similarity = 1 / (1 + distance)
        return similarity
    if similarity_type == 'pearson':
        mask = ~np.isnan(user1) & ~np.isnan(user2)
        user1_aligned = user1[mask]
        user2_aligned = user2[mask]
    
        if len(user1_aligned) == 0:  # Avoid division by zero
            return 0
    
        # Calculate Pearson correlation
        mean_user1 = np.mean(user1_aligned)
        mean_user2 = np.mean(user2_aligned)
    
        numerator = np.sum((user1_aligned - mean_user1) * (user2_aligned - mean_user2))
        denominator = np.sqrt(np.sum((user1_aligned - mean_user1) ** 2) * np.sum((user2_aligned - mean_user2) ** 2))
    
        if denominator == 0:  # Avoid division by zero
            return 0
    
        correlation = numerator / denominator
    
        # Convert correlation to similarity (if needed, optional)
        similarity = (correlation + 1) / 2  # Normalizing to [0, 1]
        return similarity
    if similarity_type == 'cosine':
        mask = ~np.isnan(user1) & ~np.isnan(user2)
        user1_aligned = user1[mask]
        user2_aligned = user2[mask]
    
        if len(user1_aligned) == 0:  # Avoid division by zero
            return 0
    
        # Calculate cosine similarity
        numerator = np.dot(user1_aligned, user2_aligned)
        denominator = np.sqrt(np.sum(user1_aligned ** 2)) * np.sqrt(np.sum(user2_aligned ** 2))
    
        if denominator == 0:  # Avoid division by zero
            return 0
    
        similarity = numerator / denominator
        return similarity


In [9]:
import time
def k_nearest_neighbors(dataset_matrix, k, similarity_type='euclidean', print_=True, print_every_n=1000):
    # Convert the pandas DataFrame to a NumPy array if it's not already an array
    if isinstance(dataset_matrix, pd.DataFrame):
        dataset_array = dataset_matrix.to_numpy()
    else:
        dataset_array = dataset_matrix

    num_users = dataset_array.shape[0]
    # Initialize the KNN matrix with zeros
    knn_matrix = np.zeros((num_users, k), dtype=int)
    similarities_matrix = np.zeros((num_users, k), dtype=float)
    
    # Placeholder for the similarity scores and user indices
    # We use np.inf as the initial values to ensure any actual similarity score will be lower
    best_similarities = np.full((k, 2), np.inf) # [similarity, index]
    
    start_total_time = time.time()  # Start timing for the whole function

    for i in range(num_users):
        start_time = time.time()  # Start timing for the current user
        for j in range(num_users):
            if i != j:
                similarity = calculate_similarity_between_users(dataset_array[i], dataset_array[j], similarity_type)
                # Check if this similarity is better than the current stored similarities
                for l in range(k-1, -1, -1):
                    if similarity < best_similarities[l, 0]:
                        if l < k - 1:
                            # Shift the later values down one position
                            best_similarities[l+1] = best_similarities[l]
                        best_similarities[l] = [similarity, j]
                    else:
                        break
                        
        # Store the indices of the K nearest neighbors
        knn_matrix[i] = best_similarities[:, 1].astype(int)
        similarities_matrix[i] = best_similarities[:, 0].astype(float)
        # Reset the best similarities for the next user
        best_similarities.fill(np.inf)

        end_time = time.time()  # End timing for the current user
        if(print_ and i % print_every_n == 0): print(f"Processed user {i+1}/{num_users} in {end_time - start_time:.2f} seconds.")
    
    end_total_time = time.time()  # End timing for the whole function
    if(print_): print(f"Total time taken: {end_total_time - start_total_time:.2f} seconds.")
    
    
    return knn_matrix, similarities_matrix


In [ ]:
#build boards for each community
train_data = []
rating_matrixes = []
knn_matrixes = []
similarities_matrixes = []
movies_map = []
for i, community in enumerate(communities):
    mask = df['userId'].isin(community)
    df_c = df[mask]
    train_data.append(df_c)
    
    matrix = create_matrix(df_c)
    movies = matrix.columns
    matrix = matrix.to_numpy()
    
    rating_matrixes.append(matrix)
    movies_map.append(movies)
    knn_matrix, similarities_matrix = k_nearest_neighbors(matrix, 3, 'pearson')
    knn_matrixes.append(knn_matrix)
    similarities_matrixes.append(similarities_matrix)

In [144]:
for i, movies in enumerate(movies_map):
    movies_map[i] = [x for x in movies]
# movies_map[0]    

In [145]:
def user_mapping(user_index, community):
    # community = find_community(user_index)
    return communities[community].index(user_index)

In [ ]:
mask = df['userId'].isin(communities[0])
df_c = df[mask]
matrix = create_matrix(df_c)

In [147]:
# def movie_mapping(movie_index, community):
#     return movies_map[community].index(movie_index)

def movie_mapping(movie_index, community):
    try:
        return movies_map[community].index(movie_index)
    except ValueError:
#         print(f"Movie index {movie_index} not found in community {community}.")
        pass


In [148]:
def predict_rating(community, user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix=True):
    # Extract the K nearest neighbors for the given user
    user_index = user_mapping(user_index, community)
    movie_index = movie_mapping(movie_index, community)
    neighbors_indices = knn_matrix[user_index]
    
    # Collect ratings for the target movie given by the neighbors
    neighbor_ratings = ratings_matrix[neighbors_indices, movie_index]
    
    # Also, collect the similarities for weighting, if applicable
    neighbor_similarities = similarities_matrix[user_index]
    
    # Filter out zeros (assuming they represent missing ratings)
    valid_indices = neighbor_ratings != 0
    valid_ratings = neighbor_ratings[valid_indices]
    valid_similarities = neighbor_similarities[valid_indices]

    
    if use_similarities_matrix and valid_ratings.size > 0:
        # Compute weighted average of valid ratings
        weighted_sum = np.sum(valid_ratings * valid_similarities)
        sum_of_weights = np.sum(valid_similarities)
        predicted_rating = weighted_sum / sum_of_weights if sum_of_weights > 0 else np.nan
    elif valid_ratings.size > 0:
        # Compute simple average of valid ratings
        predicted_rating = valid_ratings.mean()
    else:
        # If there are no valid ratings, use the global average rating
        global_average = np.nanmean(ratings_matrix[ratings_matrix != 0])
        predicted_rating = global_average
    
    return predicted_rating


In [149]:
# predict_rating(user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix=True):
def predict_rating_community(user_index, movie_index):
    community = find_community(user_index)
    return predict_rating(community, user_index, movie_index, knn_matrixes[community], similarities_matrixes[community], rating_matrixes[community])

In [150]:
from sklearn.metrics import mean_squared_error
from math import sqrt

def evaluate_model(community, knn_matrix, similarities_matrix, ratings_matrix, dataset, use_similarities_matrix = True, print_ = True, print_every_n = 5):
    
    predictions = []
    actual_ratings = []
    
    # Iterate through each row in the dataset to make predictions
    i=0
    for index, row in dataset.iterrows():
        user_index = row['userId']
        movie_index = row['movieId']
        actual_rating = row['rating']
        # Adjust indices for zero-based indexing
        if not np.isnan(actual_rating):
            
#             predicted_rating = predict_rating(user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix)
            predicted_rating = predict_rating_community(user_index, movie_index)
            #print("evaluating on value", actual_rating)
            #print("predictet value", predicted_rating)

            predictions.append(predicted_rating)
            actual_ratings.append(actual_rating)
        else:
            #print("value should be nan: ", col)
                pass
        i+=1
        if(print_ and i%print_every_n==0): print(f"done with row index: {i}, out of: {len(dataset)}")
    
    # Calculate RMSE
#     rmse = sqrt(mean_squared_error(actual_ratings, predictions))
    return (actual_ratings, predictions)


In [ ]:
def evaluate():
    actual_ratings_comm = []
    predictions_comm = []
    for community in range(len(communities)):
        print(community)
        actual_ratings, predictions = evaluate_model(community, knn_matrixes[community], similarities_matrix[community], rating_matrixes[community], train_data[community])
    #     test_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_0_fill, test_data)
        actual_ratings_comm = actual_ratings_comm + actual_ratings
        predictions_comm = predictions_comm + predictions
    print(sqrt(mean_squared_error(actual_ratings_comm, predictions_comm)))
#     print(f"Test RMSE: {test_rmse}")
#     evaluate_model(knn_matrix, similarities_matrix, ratings_matrix, dataset, use_similarities_matrix = True, print_ = Tr                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              ue, print_every_n = 500):
evaluate()

# nDCG

In [19]:
df = test_data

In [20]:
# df = pd.read_csv('/kaggle/input/knn-movielens/gt.csv')
# preds = pd.read_csv('/kaggle/input/knn-movielens/preds.csv')

In [21]:
# df.head()

In [22]:
# gt = {'user_id': [], 'item_id': [], 'rating': []}
nok = []
# preds = {'user_id': [], 'artist_id': [], 'scrobbles': []}
preds = {'userId': [], 'movieId': [], 'rating': []}
preds = pd.DataFrame(preds)
for index, row in df.iterrows():
    user_id, item_id, rating = row
    try: 
        pred_rating = predict_rating_community(user_id, item_id)
        # tmp = {'user_id': [user_id], 'artist_id': [item_id], 'scrobbles': [pred_rating]}
        tmp = {'userId': [user_id], 'movieId': [item_id], 'rating': [pred_rating]}
        tmp = pd.DataFrame(tmp)
        preds = preds._append(tmp, ignore_index=True)
    except:
        nok.append((user_id, item_id))

In [ ]:
preds.head()

In [ ]:
df.head()

In [ ]:
user_num = np.max(df['userId'])
user_num

In [26]:
df.to_csv('gt_lastfm.csv')
preds.to_csv('preds_lastfm.csv')

In [28]:
# # score = 0
# # for i in range(user_num):
#     # user_id = i+1
#     score += ndcg_score(get_top_20(df, user_id), get_top_20(preds, user_id))
# print(score/user_num)

In [29]:
def get_top_20(df, user_id):
    df_user = df[df['userId']==user_id]
    return df_user.nlargest(20, 'rating')
def get_top_10(df, user_id):
    df_user = df[df['userId']==user_id]
    return df_user.nlargest(10, 'rating')
def get_top_5(df, user_id):
    df_user = df[df['userId']==user_id]
    return df_user.nlargest(5, 'rating')

In [30]:
def ndcg_score(gt, recommended_items, k):

    temp_dcg = torch.cumsum(torch.tensor([1.0/np.log2(idx+2) if item in gt else 0 for idx, item in enumerate(recommended_items)], dtype=torch.float32), 0)

    idcg_len = min(len(gt),k)
    temp_idcg = np.cumsum(1.0 / np.log2(np.arange(2,k+2)))
    temp_idcg[idcg_len:]=temp_idcg[idcg_len-1]
    print("temp_dcg shape:", temp_dcg.shape)
    print("temp_idcg shape:", temp_idcg.shape)
    print("gt shape:", gt.shape)
    score = temp_dcg / temp_idcg
    return score

In [31]:
import torch
import numpy as np

def ndcg_score(gt, recommended_items, k):
    # Limit the recommended items to k
    recommended_items = recommended_items[:k]

    # Calculate DCG
    temp_dcg = torch.cumsum(torch.tensor(
        [1.0 / np.log2(idx + 2) if item in gt else 0 for idx, item in enumerate(recommended_items)],
        dtype=torch.float32
    ), dim=0)

    # Calculate IDCG
    ideal_items = sorted(gt, reverse=True)[:k]  # Get top k relevant items
    temp_idcg = torch.tensor(
        [1.0 / np.log2(idx + 2) for idx in range(len(ideal_items))],
        dtype=torch.float32
    )
    temp_idcg = torch.cumsum(temp_idcg, dim=0)

    # Pad temp_idcg if it has fewer elements than k
    if len(temp_dcg) < k:
        temp_dcg = torch.cat([temp_dcg, torch.zeros(k - len(temp_dcg), dtype=torch.float32)])
    if len(temp_idcg) < k:
        temp_idcg = torch.cat([temp_idcg, torch.ones(k - len(temp_idcg), dtype=torch.float32)])
    if temp_idcg.sum() == 0:
        return torch.tensor([0.0])  # or some other default value

    # Debugging shapes
#     print("temp_dcg shape:", temp_dcg.shape)
#     print("temp_idcg shape:", temp_idcg.shape)
#     print("gt shape:", gt.shape)

    # Calculate NDCG score
    score = temp_dcg / temp_idcg
    return score


In [ ]:
def recall(gt, recommended_items, k):
    recommended_items = recommended_items[:k]
    ideal_items = sorted(gt, reverse=True)[:k]
    recommended_set = set(recommended_items)
    ideal_set = set(ideal_items)
    true_positives = len(recommended_set.intersection(ideal_set))
    recall = true_positives / k

    return recall 

def get_recall_user_20(user_id):
    gt = get_top_20(df, user_id)['movieId']
    pred = get_top_20(preds, user_id)['movieId']
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return recall(gt, pred, 20)

print(get_recall_user_20(45))
       
def get_recall_user_10(user_id):
    gt = get_top_10(df, user_id)['movieId']
    pred = get_top_10(preds, user_id)['movieId']
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return recall(gt, pred, 10)

print(get_recall_user_10(45))

def get_recall_user_5(user_id):
    gt = get_top_5(df, user_id)['movieId']
    pred = get_top_5(preds, user_id)['movieId']
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return recall(gt, pred, 5)

print(get_recall_user_5(45))

In [ ]:
def get_ndcg_user_20(user_id):
    # gt = get_top_20(df, user_id)['artist_id']
    # pred = get_top_20(preds, user_id)['artist_id']
    gt = get_top_20(df, user_id)['movieId']
    pred = get_top_20(preds, user_id)['movieId']
    print(gt)
    print(pred)
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return ndcg_score(gt, pred, 20)[-1]
get_ndcg_user_20(1)

In [ ]:
def get_ndcg_user_10(user_id):
    # gt = get_top_20(df, user_id)['artist_id']
    # pred = get_top_20(preds, user_id)['artist_id']
    gt = get_top_10(df, user_id)['movieId']
    pred = get_top_10(preds, user_id)['movieId']
    print(gt)
    print(pred)
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return ndcg_score(gt, pred, 10)[-1]
get_ndcg_user_10(1)

In [ ]:
def get_ndcg_user_5(user_id):
    # gt = get_top_20(df, user_id)['artist_id']
    # pred = get_top_20(preds, user_id)['artist_id']
    gt = get_top_5(df, user_id)['movieId']
    print(gt)
    pred = get_top_5(preds, user_id)['movieId']
    print(pred)
    gt = np.reshape(gt, -1)
    pred = np.reshape(pred, -1)
    return ndcg_score(gt, pred, 5)[-1]
get_ndcg_user_5(1)

In [ ]:
recall_20 = 0
for i in range(user_num):
    user_id = i+1
    recall_20 += get_recall_user_20(user_id)
print(recall_20/user_num)
print(recall_20)
print(user_num)

In [ ]:
recall_10 = 0
for i in range(user_num):
    user_id = i+1
    recall_10 += get_recall_user_10(user_id)
print(recall_10/user_num)
print(recall_10)
print(user_num)

In [ ]:
recall_5 = 0
for i in range(user_num):
    user_id = i+1
    recall_5 += get_recall_user_5(user_id)
print(recall_5/user_num)
print(recall_5)
print(user_num)

In [ ]:
score_20 = 0
for i in range(user_num):
    user_id = i+1
    score_20 += get_ndcg_user_20(user_id)
print(score_20/user_num)
print(score_20)
print(user_num)

In [ ]:
score_10 = 0
for i in range(user_num):
    user_id = i+1
    score_10 += get_ndcg_user_10(user_id)
print(score_10/user_num)
print(score_10)
print(user_num)

In [ ]:
score_5 = 0
for i in range(user_num):
    user_id = i+1
    score_5 += get_ndcg_user_5(user_id)
print(score_5/user_num)
print(score_5)
print(user_num)

# KNN

In [42]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

In [43]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# column_names = ['user_id', 'item_id', 'rating', 'timestamp']
# df = pd.read_csv("/kaggle/input/movielens-1m-dataset/ratings.dat", sep = "::", names = column_names, engine='python')
# df.head()

In [44]:
# # Perform a train-test split
# train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# # Check the size of the resulting splits
# print(f"Train data size: {train_data.shape}")
# print(f"Test data size: {test_data.shape}")

The shape of the train set being `(80000, 4)` reflects the tabular format of the MovieLens 100K dataset, where each row represents a single rating event:
- The first number (80000) indicates the total number of individual ratings in the training dataset, derived from performing a train-test split with 20% of the data reserved for testing, leaving 80% for training.
- The second number (4) corresponds to the four columns in the dataset, which are:
  1. `'user_id'`: The ID of the user who made the rating.
  2. `'item_id'`: The ID of the item (movie) that was rated.
  3. `'rating'`: The rating value given by the user to the item.
  4. `'timestamp'`: The time at which the rating was made.

This format is different from the conceptual \(R\) matrix used in recommendation systems, where rows represent users, columns represent items, and entries represent ratings. The tabular format is more compact and practical for storing such data, as the \(R\) matrix would be very sparse (most users haven't rated most items) and inefficient to store directly.


# KNN

In [45]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import numpy as np
# import matplotlib.pyplot as plt
# import os
# import pickle


## KNN Data preperation

In [46]:
# # Load the dataset
# df = pd.read_csv('/kaggle/input/movielens-100k-dataset/ml-100k/u.data', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

# # Display the first few rows of the dataframe
# print(df.head())

In [47]:
# # Perform a train-test split
# train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

# # Check the size of the resulting splits
# print(f"Train data size: {train_data.shape}")
# print(f"Test data size: {test_data.shape}")

In [48]:
# num_users = train_data['user_id'].nunique()
# num_items = train_data['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [49]:
# num_users = test_data['user_id'].nunique()
# num_items = test_data['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [50]:
# last_user = test_data['user_id'].max()
# last_item = test_data['item_id'].max()
# print("last user id:", last_user)
# print("last item id:", last_item)

In [51]:
# train_data.head

In [52]:
# # Create a pivot table for the train data to form the user-item matrix
# train_data_matrix_0_fill = train_data.pivot_table(index='user_id', columns='item_id', values='rating', fill_value=0)

# # Display the shape and first few rows of the matrix
# print(train_data_matrix_0_fill.head())
# train_data_matrix_0_fill = train_data_matrix_0_fill.to_numpy()
# print(train_data_matrix_0_fill.shape)

The reason that the above code for prepearing data is wrong is because in the abvoe code I didn't implicitly make sure that our train_data_matrix_0_fill has an entry for each possible user and each possible item in our dataset.
As we can see:

In [53]:
# num_users = train_data['user_id'].nunique()
# num_items = train_data['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [54]:
# num_users = df['user_id'].nunique()
# num_items = df['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [55]:
# last_user = train_data['user_id'].max()
# last_item = train_data['item_id'].max()
# print("last user id:", last_user)
# print("last item id:", last_item)

# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [56]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # Assuming df is already loaded
# df_copy = df.copy()

# # Collect indices for users
# user_indices = []
# for user_id in df['user_id'].unique():
#     user_index = df_copy[df_copy['user_id'] == user_id].index[0]
#     user_indices.append(user_index)

# # Collect indices for items not already covered by user selection
# item_indices = []
# for item_id in df['item_id'].unique():
#     if not df_copy.loc[df_copy['item_id'] == item_id].index.intersection(user_indices).any():
#         item_index = df_copy[df_copy['item_id'] == item_id].index[0]
#         item_indices.append(item_index)

# # Combine indices and drop duplicates in case of overlap
# combined_indices = list(set(user_indices + item_indices))

# # Extract rows for training data
# train_data_initial = df_copy.loc[combined_indices]

# # Drop these rows from df_copy
# df_copy = df_copy.drop(index=combined_indices)

# # Perform a 75/25 split on the remaining data
# training_remaining, test_data = train_test_split(df_copy, test_size=0.25, random_state=42)

# # Combine training_remaining with the initial extracted training data
# train_data = pd.concat([train_data_initial, training_remaining]).reset_index(drop=True)

# # Output the sizes of the final splits
# print(f"Train data size: {train_data.shape}")
# print(f"Test data size: {test_data.shape}")


In [57]:
# num_users = train_data['user_id'].nunique()
# num_items = train_data['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [58]:
# num_users = df['user_id'].nunique()
# num_items = df['item_id'].nunique()
# print("number of unique users:", num_users)
# print("number of unique items:", num_items)

In [59]:
# last_user = test_data['user_id'].max()
# last_item = test_data['item_id'].max()
# print("test last user id:", last_user)
# print("test last item id:", last_item)

# last_user = df['user_id'].max()
# last_item = df['item_id'].max()
# print("test last user id:", last_user)
# print("test last item id:", last_item)

# last_user = train_data['user_id'].max()
# last_item = train_data['item_id'].max()
# print("train last user id:", last_user)
# print("train last item id:", last_item)

In [60]:
# # Create a pivot table for the train data to form the user-item matrix
# train_data_matrix_0_fill = train_data.pivot_table(index='user_id', columns='item_id', values='rating', fill_value=0)

# # Display the shape and first few rows of the matrix
# print(train_data_matrix_0_fill.head())
# train_data_matrix_0_fill = train_data_matrix_0_fill.to_numpy()
# print(train_data_matrix_0_fill.shape)

We can also do the same but use avrages instead of 0's to fill the missing values. We will use this initialization alter on to see if it makes any difference for now we will just prepare the matrix!

In [61]:
# user_rating_avg = train_data.groupby('user_id')['rating'].mean()
# user_rating_avg

In [62]:
# # Create the pivot table
# train_data_matrix = train_data.pivot_table(index='user_id', columns='item_id', values='rating')
# # Fill missing values with user averages
# train_data_matrix_avg_fill = train_data_matrix.apply(lambda x: x.fillna(user_rating_avg[x.name]), axis=1)
# # Display the shape and first few rows of the matrix
# print(train_data_matrix_avg_fill.shape)
# print(train_data_matrix_avg_fill.head())

## KNN Implamentation and Training

The following function calculates the similareties between two user entires based on their movie ratings. In this notebook we will mainly focus on the euclidian ecimilarity but later we will experiment with the cosine and paerson similareties

In [63]:
# import numpy as np

# def calculate_similarity_between_users(user1, user2, similarity_type='euclidean'):
#     if similarity_type == 'euclidean':
#         # Calculate the Euclidean distance
#         distance = np.sqrt(np.nansum((user1 - user2) ** 2))
#         # Convert distance to similarity
#         similarity = 1 / (1 + distance)
#         return similarity
#     else:
#         raise ValueError("Unsupported similarity type")


Below is the function that trains out KNN matrix. It find the k nearest neighbours based on the similareties between them. It also returns a similareties matrix that we can use for more fine tuned predictions

In [64]:
# import time
# def k_nearest_neighbors(dataset_matrix, k, similarity_type='euclidean', print_=True, print_every_n=100):
#     # Convert the pandas DataFrame to a NumPy array if it's not already an array
#     if isinstance(dataset_matrix, pd.DataFrame):
#         dataset_array = dataset_matrix.to_numpy()
#     else:
#         dataset_array = dataset_matrix

#     num_users = dataset_array.shape[0]
#     # Initialize the KNN matrix with zeros
#     knn_matrix = np.zeros((num_users, k), dtype=int)
#     similarities_matrix = np.zeros((num_users, k), dtype=float)
    
#     # Placeholder for the similarity scores and user indices
#     # We use np.inf as the initial values to ensure any actual similarity score will be lower
#     best_similarities = np.full((k, 2), np.inf) # [similarity, index]
    
#     start_total_time = time.time()  # Start timing for the whole function

#     for i in range(num_users):
#         start_time = time.time()  # Start timing for the current user
#         for j in range(num_users):
#             if i != j:
#                 similarity = calculate_similarity_between_users(dataset_array[i], dataset_array[j], similarity_type)
#                 # Check if this similarity is better than the current stored similarities
#                 for l in range(k-1, -1, -1):
#                     if similarity < best_similarities[l, 0]:
#                         if l < k - 1:
#                             # Shift the later values down one position
#                             best_similarities[l+1] = best_similarities[l]
#                         best_similarities[l] = [similarity, j]
#                     else:
#                         break
                        
#         # Store the indices of the K nearest neighbors
#         knn_matrix[i] = best_similarities[:, 1].astype(int)
#         similarities_matrix[i] = best_similarities[:, 0].astype(float)
#         # Reset the best similarities for the next user
#         best_similarities.fill(np.inf)

#         end_time = time.time()  # End timing for the current user
#         if(print_ and i % print_every_n == 0): print(f"Processed user {i+1}/{num_users} in {end_time - start_time:.2f} seconds.")
    
#     end_total_time = time.time()  # End timing for the whole function
#     if(print_): print(f"Total time taken: {end_total_time - start_total_time:.2f} seconds.")
    
    
#     return knn_matrix, similarities_matrix


In [65]:
# knn_matrix, similarities_matrix = k_nearest_neighbors(train_data_matrix_0_fill, 3, 'euclidean')

In [66]:
# train_data_matrix_avg_fill.shape

In [67]:
# import math
# Ksqrt = math.sqrt(train_data_matrix_avg_fill.shape[0])
# Ksqrt

In [68]:
# knn_matrix, similarities_matrix = k_nearest_neighbors(train_data_matrix_0_fill, int(2 * Ksqrt), 'euclidean')

In [69]:
# knn_matrix.shape

It really didn't take so long... so I don't think I'm gonna go after paralarization just yet.

## Implamenting Prediction and Evaluation 

Below is the code for the function that predicts movie ratings for users

In [70]:
# def predict_rating(user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix=True):
#     # Extract the K nearest neighbors for the given user
#     neighbors_indices = knn_matrix[user_index]
    
#     # Collect ratings for the target movie given by the neighbors
#     neighbor_ratings = ratings_matrix[neighbors_indices, movie_index]
    
#     # Also, collect the similarities for weighting, if applicable
#     neighbor_similarities = similarities_matrix[user_index]
    
#     # Filter out zeros (assuming they represent missing ratings)
#     valid_indices = neighbor_ratings != 0
#     valid_ratings = neighbor_ratings[valid_indices]
#     valid_similarities = neighbor_similarities[valid_indices]
    
#     if use_similarities_matrix and valid_ratings.size > 0:
#         # Compute weighted average of valid ratings
#         weighted_sum = np.sum(valid_ratings * valid_similarities)
#         sum_of_weights = np.sum(valid_similarities)
#         predicted_rating = weighted_sum / sum_of_weights if sum_of_weights > 0 else np.nan
#     elif valid_ratings.size > 0:
#         # Compute simple average of valid ratings
#         predicted_rating = valid_ratings.mean()
#     else:
#         # If there are no valid ratings, use the global average rating
#         global_average = np.nanmean(ratings_matrix[ratings_matrix != 0])
#         predicted_rating = global_average
    
#     return predicted_rating


Below are a bunch of outputs for insights into how our matrecies and data are structured after training

In [71]:
# knn_matrix[0, :]

In [72]:
# train_data_matrix_0_fill[knn_matrix[0], 0]

In [73]:
# train_data

In [74]:
# len(train_data)

In [75]:
# for index, row in train_data.iterrows():
#         user_index = row['user_id']
#         movie_index1=row['item_id']
#         print(movie_index1)

In [76]:
# for index, row in train_data.iterrows():
#     print(row['rating'])

Below is the rather time expensive evaluate function that evaluates the perforamnce of the current knn model on a givven dataset. We will parallize this function shortly to make it run faster in some upcomming tests

In [77]:
# from sklearn.metrics import mean_squared_error
# from math import sqrt

# def evaluate_model(knn_matrix, similarities_matrix, ratings_matrix, dataset, use_similarities_matrix = True, print_ = True, print_every_n = 500):
    
#     predictions = []
#     actual_ratings = []
    
#     # Iterate through each row in the dataset to make predictions
#     i=0
#     for index, row in dataset.iterrows():
#         user_index = row['user_id']-1
#         movie_index = row['item_id']-1
#         actual_rating = row['rating']
#         # Adjust indices for zero-based indexing
#         if not np.isnan(actual_rating):
            
#             predicted_rating = predict_rating(user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix)
#             #print("evaluating on value", actual_rating)
#             #print("predictet value", predicted_rating)

#             predictions.append(predicted_rating)
#             actual_ratings.append(actual_rating)
#         else:
#             #print("value should be nan: ", col)
#                 pass
#         i+=1
#         if(print_ and i%print_every_n==0): print(f"done with row index: {i}, out of: {len(dataset)}")
    
#     # Calculate RMSE
#     rmse = sqrt(mean_squared_error(actual_ratings, predictions))
#     return rmse


Testign the predict function

In [78]:
# import time
# st = time.time()
# predict = predict_rating(0, 0, knn_matrix, similarities_matrix, train_data_matrix_0_fill)
# end = time.time()
# print(predict, 'time:', end-st)

As we can see from above our knn predicted a rating of 4 for the user 0 and movie 0 and below we have some more insights into our data

In [79]:
# dataset_array = train_data[['user_id', 'item_id', 'rating']].to_numpy()
# dataset_array.shape

In [80]:
# print(train_data_matrix_0_fill.shape)
# print(knn_matrix.shape)

Now let's finally try evaluating our knn model

In [81]:

# train_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_0_fill, train_data)
# test_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_0_fill, test_data)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")

Training RMSE: 1.0862355732487343
Test RMSE: 1.0857690089428838

These results aren't half bad for the little training it took compared to SVD. Interestingly it's also worth noting how similar the training and test RMSE errors are which is either an indication that I did something wrong in my knn implamentationor it shows how prone to overfiting on training data other algorithems like neural network based aproaches and or SVD are!

Now lets see what happens if we don't use the use_similarities_matrix = True and set it to False. These should be a more standard "and simple" KNN implamentation.

In [82]:
# train_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_0_fill, train_data, False)
# test_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_0_fill, test_data, False)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")

A 0.0002 difference in RMSE! I guess it's noteable if you're operating on billions! I'm kidding it might just be something wrong with my implamentation or my k value or the structure of the 100k movies dataset.

Next I want to try what ahppens if we use the initialization of avrage values for missing values instead of a 0. 

*I know we prepared the matrix before and I even explicitly mentioned it but let's do it again for the sake of cohearence*

In [83]:
# # Create the pivot table
# train_data_matrix = train_data.pivot_table(index='user_id', columns='item_id', values='rating')
# # Fill missing values with user averages
# train_data_matrix_avg_fill = train_data_matrix.apply(lambda x: x.fillna(user_rating_avg[x.name]), axis=1)
# print(train_data_matrix_avg_fill.head())
# train_data_matrix_avg_fill = train_data_matrix_avg_fill.to_numpy()
# # Display the shape and first few rows of the matrix
# print(train_data_matrix_avg_fill.shape)

In [84]:
# knn_matrix, similarities_matrix = k_nearest_neighbors(train_data_matrix_avg_fill, int(2 * Ksqrt), 'euclidean')

In [85]:
# train_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_avg_fill, train_data)
# test_rmse = evaluate_model(knn_matrix, similarities_matrix, train_data_matrix_avg_fill, test_data)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")

Wow okey! I was not expacting a bigger error by 0.4. My theory is  that an avrage initialization makes the data steer towards the avrage for no real practical reason and thereby maybe it takes different nearest neighbours. Or you know.. my implamentation is wrong.

## Testing different K values

In [86]:
# from concurrent.futures import ThreadPoolExecutor
# import numpy as np
# from math import sqrt
# from sklearn.metrics import mean_squared_error
# from concurrent.futures import ProcessPoolExecutor, as_completed
# import numpy as np
# import time


Bellow is the parallized code you don't need to put much thouight to it it's there mainly for speeding up my testing times.

In [87]:
# def evaluate_model_concurrent(knn_matrix, similarities_matrix, ratings_matrix, dataset, use_similarities_matrix=True, workers=4):
#     def worker(row):
#         user_index, movie_index, actual_rating = row['user_id']-1, row['item_id']-1, row['rating']
#         if not np.isnan(actual_rating):
#             predicted_rating = predict_rating(user_index, movie_index, knn_matrix, similarities_matrix, ratings_matrix, use_similarities_matrix)
#             return predicted_rating, actual_rating
    
#     with ThreadPoolExecutor(max_workers=workers) as executor:
#         results = executor.map(worker, [row for i, row in dataset.iterrows()])
    
#     predictions, actual_ratings = zip(*[r for r in results if r is not None])
    
#     # Calculate RMSE
#     rmse = sqrt(mean_squared_error(actual_ratings, predictions))
#     return rmse


In [88]:
# import os
# import multiprocessing

# default_workers = os.cpu_count()
# default_workers


Now let's get to the juicy part. Again like with the SVD testing let's write a function that tests different k values.
In this function we don't calculate a new knn model for each k because the first k-1 values in any knn row are the same as the the whole k-1 row

In [89]:
# def test_different_ks(start_k, end_k, step, dataset_matrix, similarity_type='euclidean', use_similarities_matrix=True, train_data_portion=0.5, test_data_portion =1):
#     train_rmse_values = []
#     test_rmse_values = []
    
#     cur_train_data = train_data[:int(len(train_data)*train_data_portion)]
#     cur_test_data = train_data[:int(len(test_data)*test_data_portion)]

#     # Calculate KNN and similarities matrices for the largest k value
#     print(f"Calculating KNN and similarities matrices for k={end_k}")
#     full_knn_matrix, full_similarities_matrix = k_nearest_neighbors(dataset_matrix, end_k, similarity_type)
    
#     for k in range(start_k, end_k + 1, step):
#         print(f"Testing with k={k}")
#         start_time = time.time()
        
#         # Use only the first k columns of the precomputed matrices
#         cur_knn_matrix = full_knn_matrix[:, :k]
#         cur_similarities_matrix = full_similarities_matrix[:, :k]
        
#         train_rmse = evaluate_model(cur_knn_matrix, cur_similarities_matrix, dataset_matrix, cur_train_data, use_similarities_matrix)
#         test_rmse = evaluate_model(cur_knn_matrix, cur_similarities_matrix, dataset_matrix, cur_test_data, use_similarities_matrix)
        
#         train_rmse_values.append(train_rmse)
#         test_rmse_values.append(test_rmse)
#         end_time = time.time()
#         print(f"k = {k}, train_rmse = {train_rmse}, test_rmse = {test_rmse}, time taken = {end_time - start_time:.2f} seconds")
    
#     return full_knn_matrix, full_similarities_matrix, train_rmse_values, test_rmse_values


the parallel version of the above code that just calls the parallel evaluate function

In [90]:
# def test_different_ks_parallel(start_k, end_k, step, dataset_matrix, similarity_type='euclidean', use_similarities_matrix=True, train_data_portion=0.5, test_data_portion =1, full_knn_matrix = None, full_similarities_matrix = None):
#     cur_train_data = train_data[:int(len(train_data)*train_data_portion)]
#     cur_test_data = train_data[:int(len(test_data)*test_data_portion)]
#     train_rmse_values = []
#     test_rmse_values = []

#     #if(full_knn_matrix == None or full_similarities_matrix == None):
#        # print(f"Calculating KNN and similarities matrices for k={end_k}")
#         #full_knn_matrix, full_similarities_matrix = k_nearest_neighbors(dataset_matrix, end_k, similarity_type)
    
#     for k in range(start_k, end_k + 1, step):
#         print(f"Testing with k={k}")
#         start_time = time.time()
#         # Use only the first k columns of the precomputed matrices
#         cur_knn_matrix = full_knn_matrix[:, :k]
#         cur_similarities_matrix = full_similarities_matrix[:, :k]
        
#         train_rmse = evaluate_model_concurrent(cur_knn_matrix, cur_similarities_matrix, dataset_matrix, cur_train_data, use_similarities_matrix,workers=16)
#         test_rmse = evaluate_model_concurrent(cur_knn_matrix, cur_similarities_matrix, dataset_matrix, cur_test_data, use_similarities_matrix,workers=16)
        
#         train_rmse_values.append(train_rmse)
#         test_rmse_values.append(test_rmse)
#         end_time = time.time()
#         print(f"k = {k}, train_rmse = {train_rmse}, test_rmse = {test_rmse}")
#         print(f"time taken to test k={k} - {end_time - start_time:.2f} seconds")
        
#     return full_knn_matrix, full_similarities_matrix, train_rmse_values, test_rmse_values

In [91]:
# int(4 * Ksqrt)

In [92]:
# knn_matrix, similarities_matrix, train_rmse_values, test_rmse_values = test_different_ks(
#     start_k=1,
#     end_k=200,
#     step=1,
#     dataset_matrix=train_data_matrix_0_fill,
#     similarity_type='euclidean',
#     use_similarities_matrix=True
# )

In [93]:
# ks = range(1, int(4 * Ksqrt)+1, 1)

# plt.plot(ks, train_rmse_values, label='Train RMSE', marker='o')
# plt.plot(ks, test_rmse_values, label='Test RMSE', marker='s')
# plt.xlabel('k')
# plt.ylabel('RMSE')
# plt.title('RMSE for Different Values of k')
# plt.legend()
# plt.grid(True)
# plt.show()

As we can see from the above graph the error is sharply falling from k = 0 to k = 120

In [94]:
# knn_matrix, similarities_matrix, train_rmse_values, test_rmse_values = test_different_ks(
#     start_k=1,
#     end_k=300,
#     step=2,
#     dataset_matrix=train_data_matrix_0_fill,
#     similarity_type='euclidean',
#     use_similarities_matrix=True
# )

In [95]:
# len(train_rmse_values)

In [96]:
# ks = range(1, 300, 2)

# plt.plot(ks, train_rmse_values, label='Train RMSE', marker='o')
# plt.plot(ks, test_rmse_values, label='Test RMSE', marker='s')
# plt.xlabel('k')
# plt.ylabel('RMSE')
# plt.title('RMSE for Different Values of k')
# plt.legend()
# plt.grid(True)
# plt.show()

Lets try out different similarety methods

In [97]:
# import numpy as np
# from scipy.spatial.distance import cosine
# from scipy.stats import pearsonr

# def calculate_similarity_between_users(user1, user2, similarity_type='euclidean'):
#     # Handle missing values by using only the common non-NaN indices for comparison
#     common_indices = ~np.isnan(user1) & ~np.isnan(user2)
#     user1_common = user1[common_indices]
#     user2_common = user2[common_indices]

#     if similarity_type == 'euclidean':
#         # Calculate the Euclidean distance
#         distance = np.sqrt(np.nansum((user1_common - user2_common) ** 2))
#         # Convert distance to similarity
#         similarity = 1 / (1 + distance)
#     elif similarity_type == 'cosine':
#         # Calculate the cosine similarity. Cosine function returns the distance, so we subtract from 1.
#         similarity = 1 - cosine(user1_common, user2_common)
#     elif similarity_type == 'pearson':
#         # Calculate Pearson correlation coefficient
#         # Note: pearsonr returns a tuple (correlation coefficient, p-value). We only need the coefficient.
#         correlation, _ = pearsonr(user1_common, user2_common)
#         similarity = (correlation + 1) / 2  # Normalize to be between 0 and 1
#     else:
#         raise ValueError("Unsupported similarity type")

#     return similarity


In [98]:
# knn_matrix_cosine1, similarities_matrix_cosine1, train_rmse_values_cosine1, test_rmse_values_cosine1 = test_different_ks(
#     start_k=5,
#     end_k=10,
#     step=1,
#     dataset_matrix=train_data_matrix_0_fill,
#     similarity_type='cosine',
#     use_similarities_matrix=True
# )

In [99]:
# ks = range(5, 10+1, 1)

# plt.plot(ks, train_rmse_values_cosine1, label='Train RMSE', marker='o')
# plt.plot(ks, test_rmse_values_cosine1, label='Test RMSE', marker='s')
# plt.xlabel('k')
# plt.ylabel('RMSE')
# plt.title('RMSE for cosine similarity for Different Values of k')
# plt.legend()
# plt.grid(True)
# plt.show()

In [100]:
# knn_matrix_cosine, similarities_matrix_cosine, train_rmse_values_cosine, test_rmse_values_cosine = test_different_ks_parallel(
#     start_k=5,
#     end_k=150,
#     step=4,
#     dataset_matrix=train_data_matrix_0_fill,
#     similarity_type='cosine',
#     use_similarities_matrix=True
# )

In [101]:
# ks = range(5, 150, 4)

# plt.plot(ks, train_rmse_values_cosine, label='Train RMSE', marker='o')
# plt.plot(ks, test_rmse_values_cosine, label='Test RMSE', marker='s')
# plt.xlabel('k')
# plt.ylabel('RMSE')
# plt.title('RMSE for cosine similarity for Different Values of k')
# plt.legend()
# plt.grid(True)
# plt.show()

In [102]:
# full_knn_matrix, full_similarities_matrix = k_nearest_neighbors(train_data_matrix_0_fill, 150, 'pearson')

In [103]:

# knn_matrix_pearson, similarities_matrix_pearson, train_rmse_values_pearson, test_rmse_values_pearson = test_different_ks_parallel(
#     start_k=5,
#     end_k=150,
#     step=4,
#     dataset_matrix=train_data_matrix_0_fill,
#     similarity_type='pearson',
#     use_similarities_matrix=True,
#     full_knn_matrix=full_knn_matrix,
#     full_similarities_matrix=full_similarities_matrix
# )

In [104]:
# ks = range(5, 150, 4)

# plt.plot(ks, train_rmse_values_pearson, label='Train RMSE', marker='o')
# plt.plot(ks, test_rmse_values_pearson, label='Test RMSE', marker='s')
# plt.xlabel('k')
# plt.ylabel('RMSE')
# plt.title('RMSE for cosine similarity for Different Values of k')
# plt.legend()
# plt.grid(True)
# plt.show()

It seems that for both different similarety values the error is increasing indicating that something is wrong with my implamantation

## Adding new Users or Items or rating and adressing the cold start problem

The cold start problem is a problem when adding a new user we don't know how to intiilize his user matrix. 

Let's say we don't have the last user in our user matrix train_data_matrix_0_fill but we have his ratings. This simulates the scenario where we have a new user and we asked him to rate some movies to get a profile on him.

In [105]:
# new_user = train_data_matrix_0_fill[-1]
# # train_data_matrix_0_fill_1_missing = train_data_matrix_0_fill[:-1]
# new_id = len(train_data_matrix_0_fill)-1

In [106]:
# knn_matrix_missing, similarities_matrix_missing = k_nearest_neighbors(train_data_matrix_0_fill_1_missing, 100)
# knn_matrix_original, similarities_matrix_original = k_nearest_neighbors(train_data_matrix_0_fill, 100)

In [107]:
# predict_rating(new_id, 0, knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill)

In [108]:
# predict_rating(new_id, 0, knn_matrix_missing, similarities_matrix_missing, train_data_matrix_0_fill_1_missing)

As you can see with knn we can't really predict values for users that aren't in our system that is a downside of knn

Above we have trained oour KNN matrix without the last user. Now let's try to add the last user to our systme so we need a function that adds a new user

Here is an optimized matrix I made that adds a new user to the end of our knn matrix in O(n) time and not O(n^2) so we can use this function on our dataset each time we get and add a new user in a theoretical example in practice

In [109]:
# def add_new_user(dataset_matrix, knn_matrix, similarities_matrix, new_user, similarity_type='euclidean'):
#     k = knn_matrix.shape[1]

#     num_users = dataset_matrix.shape[0]
#     new_best_similarities = np.full((k, 2), np.inf)
#     for i in range(num_users):
#         similarity = calculate_similarity_between_users(dataset_matrix[i], new_user, similarity_type)
#         best_similarities = similarities_matrix[i]
#         # Check if this similarity is better than the current stored similarities
#         for l in range(k-1, -1, -1):
#             if similarity < best_similarities[l]:
#                 if l < k - 1:
#                     # Shift the later values down one position
#                     best_similarities[l+1] = best_similarities[l]
#                 best_similarities[l] = similarity
#             else:
#                 break
#         for l in range(k-1, -1, -1):
#             if similarity < new_best_similarities[l, 0]:
#                 if l < k - 1:
#                     # Shift the later values down one position
#                     new_best_similarities[l+1] = new_best_similarities[l]
#                 new_best_similarities[l] = [similarity, i]
#             else:
#                 break
#         similarities_matrix[i] = best_similarities

#     # Append the new user to dataset_matrix
    

#     # Prepare the arrays for the new user's nearest neighbors and their similarities
#     new_user_neighbors_indices = new_best_similarities[:, 1].astype(int)
#     new_user_similarities = new_best_similarities[:, 0]
#     # Append the new user's nearest neighbors, their similarities and new_user to the dataset_matrix
#     knn_matrix = np.vstack([knn_matrix, new_user_neighbors_indices])
#     similarities_matrix = np.vstack([similarities_matrix, new_user_similarities])
#     dataset_matrix = np.vstack([dataset_matrix, new_user])

#     return knn_matrix, similarities_matrix, dataset_matrix

    

In [110]:
# new_user

In [111]:
# updated_knn_matrix, updated_similarities_matrix, updated_matrix_0_fill = add_new_user(
#     train_data_matrix_0_fill_1_missing,
#     knn_matrix_missing,
#     similarities_matrix_missing,
#     new_user
#     )

In [112]:
# predict_rating(new_id, 0, knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill)

In [113]:
# predict_rating(new_id, 0, updated_knn_matrix, updated_similarities_matrix, updated_matrix_0_fill)

As you can see above now that we added the user into our "system" we can predict with the new user

Let's test if our system that adds a new user has the same accuracy as the original if my implamentation is correct they should be the same

In [114]:
# train_rmse = evaluate_model(knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill, train_data, print_=False)
# test_rmse = evaluate_model(knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill, test_data, print_=False)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")

In [115]:
# train_rmse = evaluate_model(updated_knn_matrix, updated_similarities_matrix, updated_matrix_0_fill, train_data, print_=False)
# test_rmse = evaluate_model(updated_knn_matrix, updated_similarities_matrix, updated_matrix_0_fill, test_data, print_=False)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")

Now we are going to simulate the scenario where we get a new item so add a new movie. TO do this we first need to add an empty movie with 0 ratings. And then we add individual ratings as entries. This is bnecause in a real scenario when we add a new movie we have 0 ratings for it because no one has rated it yet. Then we can ask users some  what they think of it and add those ratings individualy  

In [116]:
# # Simulate removing a movie column
# new_movie_index = len(train_data_matrix_0_fill[0])-1
# new_movie = train_data_matrix_0_fill[:, -1]  # Save the last column to a variable
# train_data_matrix_0_fill_less_one_movie = np.delete(train_data_matrix_0_fill, -1, axis=1)  # Remove the last column

# # Simulate adding a new movie item by adding a column of zeros to the end of the original array
# # Note: You'll want to ensure the number of rows matches for the zeros column you're adding
# num_users = train_data_matrix_0_fill_less_one_movie.shape[0]


In [117]:
# knn_matrix_missing_movie, similarities_matrix_missing_movie = k_nearest_neighbors(train_data_matrix_0_fill_less_one_movie, 100)
# knn_matrix_original, similarities_matrix_original = k_nearest_neighbors(train_data_matrix_0_fill, 100)

Now we want to split up the new_movie column into individual ratings to simulate each user rating the new movce when asked what they think about it. (This can also be thought as simulating new user ratings for any movie not necasaraly the new one added)

In [118]:
# individual_ratings = []

# # Iterate through the new_movie column to find non-zero ratings
# for user_index, rating in enumerate(new_movie):
#     if rating != 0:
#         # Append the user index and rating as a sublist
#         individual_ratings.append([user_index, new_movie_index, rating])

# print(individual_ratings)
# new_rating = individual_ratings[0]
# print(new_rating)

First let's see what happens when we try to predict.

In [119]:
# predict_rating(new_rating[0], new_rating[1], knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill)

In [120]:
# predict_rating(new_rating[0], new_rating[1], knn_matrix_missing_movie, similarities_matrix_missing_movie, train_data_matrix_0_fill_less_one_movie)


Now we need to add this rating to our Knn system similaraly to how we added a new user but just for a single rating this should also take O(n) time

In [121]:
# def alter_new_rating(dataset_matrix, knn_matrix, similarities_matrix, new_movie_ratring, similarity_type='euclidean'):
#     rating_user_index = new_movie_ratring[0]
#     rating_movie_index = new_movie_ratring[1]
#     rating = new_movie_ratring[2]
#     k = knn_matrix.shape[1]
#     altering_user = dataset_matrix[rating_user_index]
#     altering_user[rating_movie_index] = rating
#     num_users = dataset_matrix.shape[0]

#     new_best_similarities = np.full((k, 2), np.inf)
#     for i in range(num_users):

#         if i == rating_user_index: continue

#         similarity = calculate_similarity_between_users(dataset_matrix[i], altering_user, similarity_type)
#         best_similarities = similarities_matrix[i]
#         # Check if this similarity is better than the current stored similarities
#         for l in range(k-1, -1, -1):
#             if similarity < best_similarities[l]:
#                 if l < k - 1:
#                     # Shift the later values down one position
#                     best_similarities[l+1] = best_similarities[l]
#                 best_similarities[l] = similarity
#             else:
#                 break
#         for l in range(k-1, -1, -1):
#             if similarity < new_best_similarities[l, 0]:
#                 if l < k - 1:
#                     # Shift the later values down one position
#                     new_best_similarities[l+1] = new_best_similarities[l]
#                 new_best_similarities[l] = [similarity, i]
#             else:
#                 break
#         similarities_matrix[i] = best_similarities


#     # Prepare the arrays for the new user's nearest neighbors and their similarities
#     new_user_neighbors_indices = new_best_similarities[:, 1].astype(int)
#     new_user_similarities = new_best_similarities[:, 0]
#     # change the new user's nearest neighbors, their similarities and new_user to the dataset_matrix
#     knn_matrix[rating_user_index] = new_user_neighbors_indices
#     similarities_matrix[rating_user_index] = new_user_similarities
    
#     dataset_matrix[rating_user_index] = altering_user

#     return knn_matrix, similarities_matrix, dataset_matrix

    

In [122]:
# new_rating

First we add an empty column to simulate a new movie being addedwithout any ratings:

In [123]:
# zeros_column = np.zeros((num_users, 1))
# train_data_matrix_0_fill_with_new_movie = np.hstack([train_data_matrix_0_fill_less_one_movie, zeros_column])

In [124]:
# len(train_data_matrix_0_fill_less_one_movie[0])

In [125]:
# len(train_data_matrix_0_fill_with_new_movie[0])

Now we add a rating a user made for the new movie

In [126]:
# updated_knn_matrix_movie, updated_similarities_matrix_movie, updated_matrix_0_fill_movie = alter_new_rating(
#     train_data_matrix_0_fill_with_new_movie,
#     knn_matrix_missing_movie,
#     similarities_matrix_missing_movie,
#     new_rating
#     )

In [127]:
# predict_rating(new_rating[0], new_rating[1], knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill)

In [128]:
# predict_rating(new_rating[0], new_rating[1], updated_knn_matrix_movie, updated_similarities_matrix_movie, updated_matrix_0_fill_movie)

As we can see we we're able to implament adding a user and move in O(n) time 
ALso by using the alter_new_rating function we can simualte any user making a new rating and updating the system in O(n) time. 
But what if we could update a new rating in O(1)

We can use a partial update and update just the knn values of the user who made the new rating so his predictions are on point but others aren't so we treat the others like the new user never made a new rating. Then we can update the whole system in O(n^2) time. 

This aproach is usefull when we have many new ratings by many users which is the case in real life examples so we update each user in O(1) time and then when we decide durring a low trafic time for our system we update the whole system in O(n^2)  time . (During the night  for example)

In [129]:
# import numpy as np
# import random

# # Assuming train_data_matrix_0_fill_less_one_movie is your numpy matrix
# matrix = train_data_matrix_0_fill

# # Number of random non-zero ratings to select
# n = 10  # for example

# # Find all non-zero ratings along with their indices
# non_zero_indices = np.argwhere(matrix != 0)

# # Randomly select n entries from non_zero_indices
# random_selections = random.sample(list(non_zero_indices), n)

# # Create the list of [user_index, movie_index, rating] for the random selections
# random_ratings = [[user_index, movie_index, matrix[user_index, movie_index]] for user_index, movie_index in random_selections]

# print(random_ratings)


In [130]:
# missing_values_matrix_0_fill = train_data_matrix_0_fill.copy()

# for rating in random_ratings:
#     missing_values_matrix_0_fill[rating[0], rating[1]] = 0

In [131]:
# knn_matrix_missing_values, similarities_matrix_missing_values = k_nearest_neighbors(missing_values_matrix_0_fill, 100)
# knn_matrix_original, similarities_matrix_original = k_nearest_neighbors(train_data_matrix_0_fill, 100)

In [132]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill))

In [133]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], knn_matrix_missing_values, similarities_matrix_missing_values, missing_values_matrix_0_fill))

We now write the functio nthat will only alter the knn and similarety values for the altering user so he has the correct recomendations but other users are the same as if nothing changed

In [134]:
# def partly_alter_new_rating(dataset_matrix, knn_matrix, similarities_matrix, new_movie_ratring, similarity_type='euclidean'):
#     rating_user_index = new_movie_ratring[0]
#     rating_movie_index = new_movie_ratring[1]
#     rating = new_movie_ratring[2]
#     k = knn_matrix.shape[1]
#     altering_user = dataset_matrix[rating_user_index]
#     altering_user[rating_movie_index] = rating

#     #new_best_similarities = similarities_matrix[rating_user_index]
#     num_users = dataset_matrix.shape[0]

#     new_best_similarities = np.full((k, 2), np.inf)
#     for i in range(num_users):
#         if i == rating_user_index: continue

#         similarity = calculate_similarity_between_users(dataset_matrix[i], altering_user, similarity_type)
            
#         for l in range(k-1, -1, -1):
#             if similarity < new_best_similarities[l, 0]:
#                 if l < k - 1:
#                     # Shift the later values down one position
#                     new_best_similarities[l+1] = new_best_similarities[l]
#                 new_best_similarities[l] = [similarity, i]
#             else:
#                 break
        

#     # Prepare the arrays for the new user's nearest neighbors and their similarities
#     new_user_neighbors_indices = new_best_similarities[:, 1].astype(int)
#     new_user_similarities = new_best_similarities[:, 0]
#     # change the new user's nearest neighbors, their similarities and new_user to the dataset_matrix
#     knn_matrix[rating_user_index] = new_user_neighbors_indices
#     similarities_matrix[rating_user_index] = new_user_similarities
    
#     dataset_matrix[rating_user_index] = altering_user

#     return knn_matrix, similarities_matrix, dataset_matrix

    

I think we could easily optimize the abvoe function further

We will now simulate many users doing this each without knowing about the other

In [135]:
# combined_partly_updated_knn = knn_matrix_missing_values.copy()
# combined_partly_updated_similarities = similarities_matrix_missing_values.copy()
# combined_partly_updated_0_fill = missing_values_matrix_0_fill.copy()

# all_partly_updated_knn = []
# all_partly_updated_similarities = []
# all_partly_updated_0_fill = []

# for rating in random_ratings:
#     updated_knn_matrix_movie, updated_similarities_matrix_movie, updated_matrix_0_fill_movie = partly_alter_new_rating(
#     combined_partly_updated_0_fill,
#     combined_partly_updated_knn,
#     combined_partly_updated_similarities,
#     rating
#     )

#     all_partly_updated_knn.append(updated_knn_matrix_movie)
#     all_partly_updated_similarities.append(updated_similarities_matrix_movie)
#     all_partly_updated_0_fill.append(updated_matrix_0_fill_movie)

#     combined_partly_updated_knn = updated_knn_matrix_movie
#     combined_partly_updated_similarities = updated_similarities_matrix_movie
#     combined_partly_updated_0_fill = updated_matrix_0_fill_movie

In [136]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill))

In [137]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], all_partly_updated_knn[0], all_partly_updated_similarities[0], all_partly_updated_0_fill[0]))

In [138]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], combined_partly_updated_knn, combined_partly_updated_similarities, combined_partly_updated_0_fill))

Now on a not so bussy up time we can update the whole matrix for all users

In [139]:
# fully_updated_knn_matrix, fully_updated_similarities_matrix = k_nearest_neighbors(combined_partly_updated_0_fill, 100, 'euclidean')

In [140]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill))

In [141]:
# for rating in random_ratings:
#     print(predict_rating(rating[0], rating[1], fully_updated_knn_matrix, fully_updated_similarities_matrix, combined_partly_updated_0_fill))

In [142]:
# train_rmse = evaluate_model(knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill, train_data, print_=False)
# test_rmse = evaluate_model(knn_matrix_original, similarities_matrix_original, train_data_matrix_0_fill, test_data, print_=False)
# print(f"Training RMSE: {train_rmse}")
# print(f"Test RMSE: {test_rmse}")